# **STALGCM Deliverable**

## **Problem Set 2: Text Cleaning**

### **Members:**
- Jeff Uriel V. Fonte
- Ulrich Maeko L. Gonzales

In [208]:
import re
import requests
from pathlib import Path
from bs4 import BeautifulSoup

In [209]:
urls = ['https://en.wikipedia.org/wiki/2016_Philippine_presidential_election',
        'https://en.wikipedia.org/wiki/2022_Philippine_presidential_election']

In [210]:
headers = {
    'User-Agent': 'Mozilla/5.0'
    }

### **Web Scraper with BeautifulSoup**

In [211]:
def extract_text(url, headers):
    # fetch page
    page = requests.get(url, headers=headers)
    if page.status_code != 200:
        return None

    # parse page
    soup = BeautifulSoup(page.content, 'html.parser')
    body = soup.find('div', class_='mw-parser-output')
    if not body:
        return None

    # remove UI elements
    ui_elements = [
        'table', '.infobox', '.navbox',
        '.sidebar', '.hatnote', '.toc', '.catlinks'
    ]
    for element in body.select(','.join(ui_elements)):
        element.decompose()

    # extract text
    content = []
    targets = body.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'li'])

    for element in targets:
        text = element.get_text().strip()
        if text:
            content.append(text)

    return '\n\n'.join(content)

In [212]:
def save_text(file_path, text):
    # create directory if it doesn't exist
    file_path.parent.mkdir(parents=True, exist_ok=True)
    
    # write text to file
    file_path.write_text(text, encoding='utf-8')

In [213]:
for url in urls:
    year = "2016" if "2016" in url else "2022"
    
    raw_text = extract_text(url, headers)
    
    raw_path = Path(f"raw/{year}/raw_{year}_election.txt")
    save_text(raw_path, raw_text)

In [214]:
raw_files = [
    Path("raw/2016/raw_2016_election.txt"),
    Path("raw/2022/raw_2022_election.txt")
]

### **RegEx**

In [215]:
def clean_text(text):
    # Search: \[\d+\]
    # Replace: <empty>
    # Explanation: Remove citations
    text = re.sub(r"\[\d+\]", "", text, flags=re.MULTILINE)

    # Search: October 3.
    # Replace: October 3,
    # Explanation: Replace October 3. to October 3,
    text = re.sub(r"October 3.", "October 3,", text, flags=re.MULTILINE)

    # Search: Roxas, We
    # Replace: Roxas: We
    # Explanation: Replace Roxas, We with Roxas: We
    text = re.sub(r"Roxas, We", "Roxas: We", text, flags=re.MULTILINE)

    # Search: Robredo, B
    # Replace: Robredo: B
    # Explanation: Replace Robredo, B with Robredo: B
    text = re.sub(r"Robredo, B", "Robredo: B", text, flags=re.MULTILINE)

    # Search: Jose Montemayor Jr. (DPP)
    # Replace: <empty>
    # Explanation: Remove the specific name and title
    text = re.sub(r"Jose Montemayor Jr\. \(DPP\)\n", "", text, flags=re.MULTILINE)

    # Search: \[.*\]
    # Replace: <empty>
    # Explanation: Remove any remaining brackets
    text = re.sub(r"\[.*\]", "", text, flags=re.MULTILINE)

    # Search: permanent dead link
    # Replace: <empty>
    # Explanation: Remove notes
    text = re.sub(r"permanent dead link", "", text, flags=re.MULTILINE)

    # Search: ^[^.\n]+$
    # Replace: <empty>
    # Explanation: Remove headings, but avoid colons
    text = re.sub(r"^(?!.*:)[^.\n]+$", "", text, flags=re.MULTILINE)

    # Search: ^\s*\n
    # Replace: <empty>
    # Explanation: Remove empty lines
    text = re.sub(r"^\s*\n", "", text, flags=re.MULTILINE)

    # Search: ^(↑.*)$
    # Replace: [\1]
    # Explanation: Wrap lines starting with ↑ and '1 2' in brackets
    text = re.sub(r"^(\↑.*|1 2.*)$", r"[\1]", text, flags=re.MULTILINE)

    # Search: (?<!↑)(?<!Jr\.)(?<!Dr\.)(?<!Mr\.)(?<!Sen\.)(?<!Brgy\.)(?<!U\.S\.)(?<!transl\.)(?<!lit\.)(?<=\.)\s+
    # Replace: \n
    # Explanation: Segment sentences
    text = re.sub(r"(?<!Jr\.)(?<!Dr\.)(?<!Mr\.)(?<!Sen\.)(?<!Brgy\.)(?<!U\.S\.)(?<!transl\.)(?<!lit\.)(?<=\.)(?![^\[]*\])\s+", "\n", text, flags=re.MULTILINE)

    # Search: [\“\”\„\”]
    # Replace: \"
    # Explanation: Replace quotation marks with regular double quotation mark
    text = re.sub(r"[\“\”\„\”]", '"', text)

    # Search: (?<=\.\")\s+(?=[A-Z\"])
    # Replace: \n
    # Explanation: Segment sentences with quotations
    text = re.sub(r"(?<=\.\")(?![^\[]*\])\s+(?=[A-Z\"])", "\n", text)

    # Search: 
    # Replace: 
    # Explanation: Remove brackets
    text = re.sub(r"[\[\]]", "", text)

    # Search: 
    # Replace: 
    # Explanation: Remove brackets
    text = re.sub(r"\↑ |1 2 3 |1 2 |", "", text)

    # Search: 
    # Replace: 
    # Explanation: Remove brackets
    text = re.sub(r"Notes:\n", "", text)

    return text

In [216]:
for raw_path in raw_files:
    year = "2016" if "2016" in str(raw_path) else "2022"
    
    raw_text = raw_path.read_text(encoding='utf-8')
    
    processed_text = clean_text(raw_text)
    
    processed_path = Path(f"processed/processed_{year}_election.txt")
    save_text(processed_path, processed_text)